In [ ]:
import pandas as pd
from drillcore_transformations.transformations import transform
from pathlib import Path
import bisect
from functools import partial
from itertools import starmap

In [ ]:
MEASUREMENTS_PATH = Path("src/drillcore_transformations/tests/sample_data/measurement_sample.csv")
DEPTHS_PATH = Path("src/drillcore_transformations/tests/sample_data/depth_sample.csv")

In [ ]:
measurements = pd.read_csv(MEASUREMENTS_PATH, sep=";")
depths = pd.read_csv(DEPTHS_PATH, sep=";")

In [ ]:
measurements

In [ ]:
depths

In [ ]:
depths.plot(x="DEPTH", y="DIP")

In [ ]:
def resolve_drillcore_trend_plunge(depth:float, depths:pd.DataFrame, depth_column="DEPTH", trend_column:str = "AZIMUTH", plunge_column:str = "DIP")-> tuple[float, float]:
    # Takes the value to the "left". I.e. if depths are sorted ascending, it takes the orientation value of the lower depth
    # E.g. at depth of 42.6 m, with orientation defined at 40 m and 45 m, orientation at 40 m will be used
    bisect_index = bisect.bisect_left(depths[depth_column], depth) - 1
    trend = depths[trend_column].iloc[bisect_index]
    plunge = depths[plunge_column].iloc[bisect_index]
    return trend, plunge

In [ ]:
drillcore_trend, drillcore_plunge = zip(*measurements["LENGTH_FROM"].apply(partial(resolve_drillcore_trend_plunge, depths=depths)).to_numpy())

In [ ]:
measurements["drillcore_trend"] = drillcore_trend
measurements["drillcore_plunge"] = drillcore_plunge

In [ ]:
measurements

In [ ]:
dips, directions, _, _ = zip(*starmap(transform, measurements[["ALPHA_CORE", "BETA_CORE", "drillcore_trend", "drillcore_plunge"]].to_numpy()))

In [ ]:
measurements["dip"] = dips
measurements["direction"] = directions

In [ ]:
measurements[["ALPHA_CORE", "BETA_CORE", "dip", "direction", "drillcore_trend", "drillcore_plunge"]]